In [ ]:
!pip install great_expectations --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv('produtos_hardware_eletronicos.csv')

print("Total de linhas:", len(df))
print("\nColunas:", list(df.columns))
print("\nValores nulos por coluna:")
print(df.isnull().sum())

Total de linhas: 118338

Colunas: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth', 'category_name']

Valores nulos por coluna:
asin                 0
title                0
imgUrl               0
productURL           0
stars                0
reviews              0
price                0
listPrice            0
category_id          0
isBestSeller         0
boughtInLastMonth    0
category_name        0
dtype: int64


In [ ]:
# Verificações de qualidade específicas

print("=== Preço ===")
print("Produtos com preço = 0 (sem preço válido):", (df['price'] == 0).sum())
print("Produtos com preço negativo:", (df['price'] < 0).sum())

print("\n=== Avaliação (stars) ===")
print("Produtos sem avaliação (stars = 0):", (df['stars'] == 0).sum())
print("Produtos com nota fora do intervalo 1-5:", ((df['stars'] < 0) | (df['stars'] > 5)).sum())

print("\n=== Duplicados ===")
print("Produtos com ASIN duplicado:", df['asin'].duplicated().sum())

print("\n=== Título vazio ou muito curto ===")
print("Títulos vazios:", (df['title'].str.strip() == '').sum())
print("Títulos com menos de 5 caracteres:", (df['title'].str.len() < 5).sum())

=== Preço ===
Produtos com preço = 0 (sem preço válido): 4362
Produtos com preço negativo: 0

=== Avaliação (stars) ===
Produtos sem avaliação (stars = 0): 16594
Produtos com nota fora do intervalo 1-5: 0

=== Duplicados ===
Produtos com ASIN duplicado: 0

=== Título vazio ou muito curto ===
Títulos vazios: 0
Títulos com menos de 5 caracteres: 5


In [ ]:
import pandas as pd
df = pd.read_csv('produtos_hardware_eletronicos.csv')

In [ ]:
import great_expectations as gx

# Transforma o DataFrame num objeto validável pelo great-expectations
df_ge = gx.from_pandas(df)

# Regra 1: preço não pode ser zero (produto sem preço válido)
resultado_preco = df_ge.expect_column_values_to_be_between(
    column="price", min_value=0.01, max_value=None
)

# Regra 2: avaliação (stars) deve estar entre 1 e 5
resultado_stars = df_ge.expect_column_values_to_be_between(
    column="stars", min_value=1, max_value=5
)

# Regra 3: ASIN não pode ter duplicado (cada produto é único)
resultado_asin = df_ge.expect_column_values_to_be_unique(column="asin")

# Regra 4: título deve ter pelo menos 5 caracteres
resultado_titulo = df_ge.expect_column_value_lengths_to_be_between(
    column="title", min_value=5, max_value=None
)

# Resumo dos resultados
print("Preço válido (>0):", resultado_preco["success"], "-", resultado_preco["result"]["unexpected_count"], "produtos falharam")
print("Avaliação entre 1-5:", resultado_stars["success"], "-", resultado_stars["result"]["unexpected_count"], "produtos falharam")
print("ASIN único:", resultado_asin["success"], "-", resultado_asin["result"]["unexpected_count"], "produtos falharam")
print("Título com 5+ caracteres:", resultado_titulo["success"], "-", resultado_titulo["result"]["unexpected_count"], "produtos falharam")

  return datetime.utcnow().replace(tzinfo=utc)



Preço válido (>0): False - 4362 produtos falharam
Avaliação entre 1-5: False - 16594 produtos falharam
ASIN único: True - 0 produtos falharam
Título com 5+ caracteres: False - 5 produtos falharam
